In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Sequence
import os 
import glob 
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.colors import Normalize
from matplotlib.colors import BoundaryNorm, ListedColormap, LinearSegmentedColormap


In [2]:
@dataclass(frozen=True)
class S2SSkillPlotConfig:
    """Plot config for s2s_skill_<GROUP>_<FREQ>_<RUN>_<OBS>_<VAR>.nc style files."""
    skill_dir: Path
    file_glob: str = "s2s_skill_*_*.nc"

    # What to plot
    vars_include: tuple[str, ...] = ()   # if empty -> all variables found from filenames
    exp_include: tuple[str, ...] = ()    # if empty -> all exps in file
    exp_order: tuple[str, ...] = ()      # optional explicit ordering

    # Lead bins (days since init); use lead_days coordinate
    lead_bins: tuple[tuple[int, int], ...] = ((1, 14), (15, 28), (29, 42), (43, 56))
    lead_bin_labels: tuple[str, ...] = ("Wk1–2", "Wk3–4", "Wk5–6", "Wk7–8")

    # Curves
    smooth_days: int = 7  # 0 or 1 = no smoothing
    time_max: int | None = None  # optionally cap lead days for curve

    # Bars
    bar_metric: str = "ACC"       # "ACC" or "RMSE"
    bar_use_quantiles: bool = True  # if available, use quantile whiskers; else std

    # Line plot meaning (must be explicit)
    line_type: str = "ens_mean"
    # options:
    # "ens_mean"        -> line = metric
    # "member_mean"     -> line = metric_member_mean
    # "ens_mean_std"    -> line = metric, shading = ± metric_member_std
    # "member_mean_std" -> line = metric_member_mean, shading = ± metric_member_std
    # "ens_vs_member"   -> two lines: metric (solid) and metric_member_mean (dashed)

    # Style
    figsize_curve: tuple[float, float] = (10.5, 4.2)
    figsize_bar: tuple[float, float] = (11.5, 4.8)
    legend_ncol: int = 4

    # Output
    out_dir: Path | None = None
    dpi: int = 300

    # Optional: tag to avoid overwriting files when you change groups
    tag: str = ""

    # ---- styling controls ----
    exp_colors: Mapping[str, str] | None = None
    xlim: tuple[float, float] | None = None
    ylim: tuple[float, float] | None = None


class S2SSkillPlotter:
    """
    Reads s2s_skill_*.nc files with dims:
      time, exp, var(=1), ens, q(=3)

    vars:
      ACC(var, exp, time), RMSE(...)
      ACC_member_std(var, exp, time), ...
      ACC_member_mean(var, exp, time), ...
      ACC_member_quantile(q, var, exp, time), ...
      lead_days(time)
    """

    def __init__(self, cfg: S2SSkillPlotConfig):
        self.cfg = cfg
        if self.cfg.out_dir:
            self.cfg.out_dir.mkdir(parents=True, exist_ok=True)

    # ---------- IO helpers ----------
    def _discover_files(self) -> list[Path]:
        files = sorted(self.cfg.skill_dir.glob(self.cfg.file_glob))
        if not files:
            raise FileNotFoundError(f"No files matched {self.cfg.skill_dir}/{self.cfg.file_glob}")
        return files

    @staticmethod
    def _var_from_filename(p: Path) -> str:
        # expects ..._<VAR>.nc
        return p.stem.split("_")[-1]

    def _select_exps(self, ds: xr.Dataset) -> xr.Dataset:
        if "exp" not in ds:
            return ds

        exps = [str(e) for e in ds["exp"].values.tolist()]

        if self.cfg.exp_include:
            keep = [e for e in exps if e in set(self.cfg.exp_include)]
        else:
            keep = exps

        if self.cfg.exp_order:
            keep = [e for e in self.cfg.exp_order if e in set(keep)]

        if not keep:
            raise ValueError(
                "No experiments selected. Check exp_order/exp_include vs file's exp names. "
                f"File exps={exps}, requested exp_order={list(self.cfg.exp_order)}"
            )

        return ds.sel(exp=keep)

    # ---------- lead/rolling helpers ----------
    @staticmethod
    def _get_lead_days(ds: xr.Dataset) -> xr.DataArray:
        if "lead_days" in ds:
            lead = ds["lead_days"]
        else:
            lead = xr.DataArray(np.arange(ds.sizes["time"]), dims=("time",), name="lead_days")
        return lead.astype(float)

    @staticmethod
    def _rolling_mean(da: xr.DataArray, win: int) -> xr.DataArray:
        if win is None or win <= 1:
            return da
        return da.rolling(time=win, center=True, min_periods=1).mean()

    def _maybe_cap_time(self, ds: xr.Dataset) -> xr.Dataset:
        if self.cfg.time_max is None:
            return ds
        lead = self._get_lead_days(ds)
        return ds.sel(time=lead <= float(self.cfg.time_max))

    # ---------- binning helpers ----------
    def _bin_mask(self, lead: xr.DataArray, a: int, b: int) -> xr.DataArray:
        return (lead >= float(a)) & (lead <= float(b))

    def _bin_mean(self, da: xr.DataArray, lead: xr.DataArray, a: int, b: int) -> xr.DataArray:
        m = self._bin_mask(lead, a, b)
        return da.where(m).mean("time", skipna=True)

    # ---------- quantile parsing ----------
    @staticmethod
    def _infer_quantile_levels(qvals: np.ndarray) -> tuple[int, int, int]:
        qvals = np.array(qvals, dtype=float)
        order = np.argsort(qvals)
        low = int(order[0])
        high = int(order[-1])
        mid = int(order[np.argmin(np.abs(qvals[order] - 0.5))])
        return low, mid, high

    # ----------------------------
    # Public API
    # ----------------------------
    def open_skill(self, var_name: str) -> xr.Dataset:
        files = self._discover_files()
        candidates = [p for p in files if self._var_from_filename(p) == var_name]
        if not candidates:
            raise FileNotFoundError(
                f"No file found for var={var_name} under {self.cfg.skill_dir} with glob={self.cfg.file_glob}"
            )
    
        # Open lazily, subset, then load and close to avoid file-handle leaks
        with xr.open_dataset(candidates[0]) as ds0:
            ds = ds0.copy(deep=False)     # cheap view first
            ds = self._select_exps(ds)
            ds = self._maybe_cap_time(ds)
    
            # only load needed variables (keep it small)
            keep = [v for v in ds.data_vars if v.startswith(("ACC", "RMSE"))] + ["lead_days"]
            keep = [v for v in keep if v in ds]
            ds = ds[keep].load()          # bring into memory, safe after file closes
    
        return ds

    def list_vars(self) -> list[str]:
        files = self._discover_files()
        vars_ = [self._var_from_filename(p) for p in files]
        if self.cfg.vars_include:
            vars_ = [v for v in vars_ if v in set(self.cfg.vars_include)]
        return sorted(dict.fromkeys(vars_))

    def _select_line_and_spread(self, ds: xr.Dataset, metric: str):
        lt = self.cfg.line_type

        if lt == "ens_mean":
            return ds[metric], None, f"{metric} (ens mean)", False

        if lt == "member_mean":
            return ds[f"{metric}_member_mean"], None, f"{metric} (member mean)", False

        if lt == "ens_mean_std":
            return ds[metric], ds.get(f"{metric}_member_std"), f"{metric} (ens mean ± member std)", False

        if lt == "member_mean_std":
            return ds[f"{metric}_member_mean"], ds.get(f"{metric}_member_std"), f"{metric} (member mean ± std)", False

        if lt == "ens_vs_member":
            return None, None, f"{metric} (ens vs member)", True

        raise ValueError(f"Unknown line_type: {lt}")

    def _out_name(self, kind: str, var_name: str, metric: str | None = None) -> str:
        tag = f"_{self.cfg.tag}" if self.cfg.tag else ""
        met = f"_{metric}" if metric else ""
        return f"s2s_{kind}_{var_name}{met}_{self.cfg.line_type}{tag}.png"

    def _get_exp_color(self, exp: str, i: int) -> str:
        if self.cfg.exp_colors and exp in self.cfg.exp_colors:
            return self.cfg.exp_colors[exp]
        return plt.rcParams["axes.prop_cycle"].by_key()["color"][i % 10]

    def _apply_axes_limits(self, ax: plt.Axes) -> None:
        if self.cfg.xlim is not None:
            ax.set_xlim(*self.cfg.xlim)
        if self.cfg.ylim is not None:
            ax.set_ylim(*self.cfg.ylim)

    def _weekbin_matrix(self, metric: str, a: int, b: int) -> tuple[np.ndarray, list[str], list[str]]:
        """
        Returns:
          M: (nvar, nexp) matrix of bin-mean metric
          vars_: list of variable names in row order
          exps_: list of experiment names in column order
        """
        vars_ = self.list_vars()
        vars_ = [v for v in vars_ if (not self.cfg.vars_include) or (v in set(self.cfg.vars_include))]

        first = self.open_skill(vars_[0])
        exps_ = [str(e) for e in first["exp"].values.tolist()]

        M = np.full((len(vars_), len(exps_)), np.nan, dtype=float)

        for i, v in enumerate(vars_):
            ds = self.open_skill(v)
            lead = self._get_lead_days(ds)

            if metric not in ds:
                continue

            da = ds[metric]
            if "var" in da.dims:
                da = da.isel(var=0)

            for j, e in enumerate(exps_):
                if e not in da["exp"].values:
                    continue
                M[i, j] = float(self._bin_mean(da.sel(exp=e), lead, a, b).values)

        return M, vars_, exps_

    # ---------- plots ----------
    def plot_skill_curves(
        self,
        var_name: str,
        metrics: Sequence[str] = ("ACC",),
        out_png: Path | None = None,
        show: bool = True,
        fontz: int = 12, 
    ) -> plt.Figure:
        ds = self.open_skill(var_name)
        lead = self._get_lead_days(ds)

        fig, ax = plt.subplots(figsize=self.cfg.figsize_curve)
        plotted_any = False

        for met in metrics:
            if met not in ds and f"{met}_member_mean" not in ds:
                continue

            line_da, spread_da, label, do_both = self._select_line_and_spread(ds, met)

            # ens_vs_member: two lines (solid = ens mean, dashed = member mean)
            if do_both:
                if met not in ds or f"{met}_member_mean" not in ds:
                    continue

                da_ens = ds[met]
                da_mem = ds[f"{met}_member_mean"]

                if "var" in da_ens.dims:
                    da_ens = da_ens.isel(var=0)
                if "var" in da_mem.dims:
                    da_mem = da_mem.isel(var=0)

                da_ens = self._rolling_mean(da_ens, self.cfg.smooth_days)
                da_mem = self._rolling_mean(da_mem, self.cfg.smooth_days)

                for i, exp in enumerate(da_ens["exp"].values):
                    exp = str(exp)
                    color = self._get_exp_color(exp, i)
                    ax.plot(
                        lead.values,
                        da_ens.sel(exp=exp).values,
                        label=f"{exp}",
                        linewidth=2,
                        color=color,
                    )
                    ax.plot(
                        lead.values,
                        da_mem.sel(exp=exp).values,
                        label=f"{exp}",
                        linewidth=1.7,
                        linestyle="--",
                        color=color,
                    )

                plotted_any = True
                continue

            # Single line (+ optional shading)
            da_line = line_da
            if "var" in da_line.dims:
                da_line = da_line.isel(var=0)
            da_line = self._rolling_mean(da_line, self.cfg.smooth_days)

            da_spread = None
            if spread_da is not None:
                da_spread = spread_da
                if "var" in da_spread.dims:
                    da_spread = da_spread.isel(var=0)
                da_spread = self._rolling_mean(da_spread, self.cfg.smooth_days)

            for i, exp in enumerate(da_line["exp"].values):
                exp = str(exp)
                y = da_line.sel(exp=exp)
                color = self._get_exp_color(exp, i)
                ax.plot(lead.values, y.values, label=f"{exp} {label}", linewidth=2, color=color)

                if da_spread is not None:
                    s = da_spread.sel(exp=exp)
                    ax.fill_between(lead.values, (y - s).values, (y + s).values, alpha=0.2, color=color)

            plotted_any = True

        if not plotted_any:
            plt.close(fig)
            raise ValueError(f"No requested metrics available to plot for var={var_name}. metrics={metrics}")

        ax.axhline(0.0, linewidth=1.0)
        ax.set_title(
            f"{var_name}: {self.cfg.line_type}, smooth={self.cfg.smooth_days}d",
            fontsize = fontz*1.0
        )

        if len(metrics) == 1:
            ax.set_ylabel(
                "Skill (ACC)" if metrics[0].upper() == "ACC" else "Error (RMSE)", 
                fontsize=fontz * 0.95, labelpad=6
            )
        else:
            ax.set_ylabel("Skill / Error", fontsize=fontz * 0.95, labelpad=6)
            
        ax.set_xlabel("Lead day since init", fontsize=fontz * 0.95, labelpad=6)
        
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )
        self._apply_axes_limits(ax)

        ax.legend(ncol=self.cfg.legend_ncol, fontsize=fontz*0.9)
        ax.grid(True, alpha=0.25)

        if out_png is None and self.cfg.out_dir:
            out_png = self.cfg.out_dir / self._out_name(
                "curve", var_name, metric=metrics[0] if len(metrics) == 1 else None
            )
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig

    def plot_week_binned_bars(
        self,
        var_name: str,
        metric: str | None = None,
        out_png: Path | None = None,
        show: bool = True,
        fontz: int = 12, 
    ) -> plt.Figure:
        metric = metric or self.cfg.bar_metric
        ds = self.open_skill(var_name)
        lead = self._get_lead_days(ds)

        if metric not in ds:
            raise KeyError(f"{metric} not found in dataset for {var_name}")

        da = ds[metric]
        if "var" in da.dims:
            da = da.isel(var=0)

        exps = [str(e) for e in da["exp"].values.tolist()]
        nb = len(self.cfg.lead_bins)
        x = np.arange(nb)

        fig, ax = plt.subplots(figsize=self.cfg.figsize_bar)
        width = 0.8 / max(len(exps), 1)

        q_da = None
        std_da = None
        if self.cfg.bar_use_quantiles:
            qname = f"{metric}_member_quantile"
            if qname in ds:
                q_da = ds[qname]
                if "var" in q_da.dims:
                    q_da = q_da.isel(var=0)
        if q_da is None:
            sname = f"{metric}_member_std"
            if sname in ds:
                std_da = ds[sname]
                if "var" in std_da.dims:
                    std_da = std_da.isel(var=0)

        q_idx = None
        if q_da is not None and "q" in q_da.dims:
            q_idx = self._infer_quantile_levels(q_da["q"].values)

        for i, exp in enumerate(exps):
            means, yerr_low, yerr_high = [], [], []

            for (a, b) in self.cfg.lead_bins:
                m = self._bin_mean(da.sel(exp=exp), lead, a, b).item()
                means.append(m)

                if q_da is not None and q_idx is not None:
                    lowi, midi, highi = q_idx
                    qbin = self._bin_mean(q_da.sel(exp=exp), lead, a, b)
                    qlow = float(qbin.isel(q=lowi).values)
                    qmid = float(qbin.isel(q=midi).values)
                    qhigh = float(qbin.isel(q=highi).values)
                    yerr_low.append(abs(qmid - qlow))
                    yerr_high.append(abs(qhigh - qmid))
                elif std_da is not None:
                    s = float(self._bin_mean(std_da.sel(exp=exp), lead, a, b).values)
                    yerr_low.append(s)
                    yerr_high.append(s)
                else:
                    yerr_low.append(0.0)
                    yerr_high.append(0.0)

            means = np.array(means, dtype=float)
            yerr = np.vstack([np.array(yerr_low), np.array(yerr_high)])

            xpos = x + (i - (len(exps) - 1) / 2) * width
            color = self._get_exp_color(exp, i)
            ax.bar(xpos, means, width=width, label=exp, color=color)
            ax.errorbar(xpos, means, yerr=yerr, fmt="none", capsize=3, linewidth=1.0, color=color)

        ax.set_xticks(x)
        ax.set_xticklabels(self.cfg.lead_bin_labels)
        
        ax.set_title(
            f"{var_name}: {metric} week-binned (member uncertainty)",
            fontsize = fontz*1.0
        )

        ax.set_ylabel(metric, fontsize=fontz * 0.95, labelpad=6)
        #ax.set_xlabel("Lead day since init", fontsize=fontz * 0.95, labelpad=6)
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )

        ax.grid(True, axis="y", alpha=0.25)
        ax.legend(ncol=self.cfg.legend_ncol, fontsize=fontz*0.9)

        self._apply_axes_limits(ax)

        if out_png is None and self.cfg.out_dir:
            out_png = self.cfg.out_dir / self._out_name("weekbins", var_name, metric=metric)
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig

    def plot_shock_metric(
        self,
        var_name: str,
        metric: str = "ACC",
        early: tuple[int, int] = (1, 3),
        late: tuple[int, int] = (8, 14),
        out_png: Path | None = None,
        show: bool = True,
    ) -> plt.Figure:
        ds = self.open_skill(var_name)
        lead = self._get_lead_days(ds)

        da = ds[metric]
        if "var" in da.dims:
            da = da.isel(var=0)

        exps = [str(e) for e in da["exp"].values.tolist()]
        shock = []
        for exp in exps:
            m1 = float(self._bin_mean(da.sel(exp=exp), lead, early[0], early[1]).values)
            m2 = float(self._bin_mean(da.sel(exp=exp), lead, late[0], late[1]).values)
            shock.append(m1 - m2)

        fig, ax = plt.subplots(figsize=(8.8, 3.8))
        for i, exp in enumerate(exps):
            ax.bar(i, shock[i], color=self._get_exp_color(exp, i))
        ax.axhline(0.0, linewidth=1.0)
        ax.set_xticks(np.arange(len(exps)))
        ax.set_xticklabels(exps, rotation=25, ha="right")
        
        ax.set_title(
            f"{var_name}: shock ({metric} {early[0]}–{early[1]} minus {late[0]}–{late[1]})",
            fontsize = fontz*1.0
        )
        ax.set_ylabel("Shock (higher = less early collapse)", fontsize=fontz * 0.95, labelpad=6)
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )
        
        ax.grid(True, axis="y", alpha=0.25)

        self._apply_axes_limits(ax)

        if out_png is None and self.cfg.out_dir:
            out_png = self.cfg.out_dir / self._out_name("shock", var_name, metric=metric)
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig

    def plot_weekbin_heatmap(
        self,
        metric: str = "ACC",
        bin_index: int = 0,
        out_png: Path | None = None,
        show: bool = True,
        vmin: float | None = None,
        vmax: float | None = None,
        cmap: str = "coolwarm",          # <- like your example
        center: float | None = None,   # if set, uses symmetric limits around center
        annotate: bool = True,
        fmt: str = "{:.2f}",
        annot_fs: int = 7,
        tick_fs: int = 8,
        grid_lw: float = 0.6,
        xlabel: str = "Variable",
        ylabel: str = "Experiment",
    ) -> plt.Figure:
        """
        Styled single-bin heatmap:
          rows = experiments
          cols = variables
          values = bin-mean(metric) over lead_bins[bin_index]
        """
        a, b = self.cfg.lead_bins[bin_index]
        label = self.cfg.lead_bin_labels[bin_index]
    
        # M: (nvar, nexp). We plot M.T -> (nexp, nvar)
        M, vars_, exps_ = self._weekbin_matrix(metric, a, b)
        Z = M.T  # rows=exp, cols=var
    
        # infer limits
        finite = Z[np.isfinite(Z)]
        if finite.size == 0:
            raise ValueError(f"All values are NaN for metric={metric} bin={label}.")
    
        if center is not None:
            # symmetric around center, like many skill heatmaps
            if vmin is None or vmax is None:
                d = float(np.nanmax(np.abs(finite - center)))
                vmin, vmax = center - d, center + d
        else:
            if vmin is None:
                vmin = float(np.nanmin(finite))
            if vmax is None:
                vmax = float(np.nanmax(finite))
    
        # figsize tuned for many variables (like your example)
        fig_w = max(10.0, 0.45 * len(vars_) + 2.5)
        fig_h = max(2.6, 0.55 * len(exps_) + 1.2)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    
        # draw as pcolormesh to get crisp borders
        x = np.arange(len(vars_) + 1)
        y = np.arange(len(exps_) + 1)
        pm = ax.pcolormesh(
            x, y, Z,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            shading="flat",
            edgecolors="white",
            linewidth=grid_lw,
        )
    
        # ticks at cell centers
        ax.set_xticks(np.arange(len(vars_)) + 0.5)
        ax.set_yticks(np.arange(len(exps_)) + 0.5)
        ax.set_xticklabels(vars_, rotation=30, ha="right", fontsize=tick_fs)
        ax.set_yticklabels(exps_, fontsize=tick_fs)
    
        ax.invert_yaxis()  # first exp on top (like your example)
        ax.set_xlabel(xlabel, fontsize=fontz * 0.95, labelpad=6)
        ax.set_ylabel(ylabel, fontsize=fontz * 0.95, labelpad=6)
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )
        
        ax.set_title(
            f"{metric} ({label}: lead {a:02d}–{b:02d} days)",
            fontsize = fontz*1.0
        )

        # annotate
        if annotate:
            for i in range(Z.shape[0]):      # exp row
                for j in range(Z.shape[1]):  # var col
                    val = Z[i, j]
                    if not np.isfinite(val):
                        continue
                    ax.text(
                        j + 0.5, i + 0.5,
                        fmt.format(val),
                        ha="center", va="center",
                        fontsize=annot_fs,
                        color="black",
                    )
    
        # colorbar
        cbar = fig.colorbar(
            sm, ax=ax, shrink=0.9, pad=0.02,
            boundaries=bounds, ticks=bounds
        )
        cbar.ax.tick_params(labelsize=8)
        cbar.set_label(metric)
    
        fig.tight_layout()
    
        if out_png is None and self.cfg.out_dir:
            tag = f"_{self.cfg.tag}" if self.cfg.tag else ""
            out_png = self.cfg.out_dir / f"s2s_heatmap_{metric}_{label}{tag}.png"
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")
    
        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig


    def plot_weekbin_heatmap_diff(
        self,
        metric: str = "ACC",
        bin_index: int = 0,
        baseline_exp: str = "CTRL10-S0",
        out_png: Path | None = None,
        show: bool = True,
        vabs: float | None = None,
        xlabel: str = "Variable",
        ylabel: str = "Experiment",
    ) -> plt.Figure:
        a, b = self.cfg.lead_bins[bin_index]
        label = self.cfg.lead_bin_labels[bin_index]

        M, vars_, exps_ = self._weekbin_matrix(metric, a, b)

        if baseline_exp not in exps_:
            raise ValueError(f"baseline_exp={baseline_exp} not found in exps: {exps_}")

        j0 = exps_.index(baseline_exp)
        D = M - M[:, [j0]]

        if vabs is not None:
            vmin, vmax = -float(vabs), float(vabs)
        else:
            vmin = vmax = None

        fig, ax = plt.subplots(figsize=(1.2 * len(vars_) + 5.0, 0.6 * len(exps_) + 2.0))
        im = ax.imshow(D.T, aspect="auto", vmin=vmin, vmax=vmax)  # transpose: rows=exp, cols=var

        ax.set_title(f"{metric} diff heatmap vs {baseline_exp} ({label}: {a}–{b} days)")
        ax.set_xlabel(xlabel, fontsize=fontz * 0.95, labelpad=6)
        ax.set_ylabel(ylabel, fontsize=fontz * 0.95, labelpad=6)
        
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )
        
        ax.set_xticks(np.arange(len(vars_)))
        ax.set_xticklabels(vars_, rotation=30, ha="right")

        ax.set_yticks(np.arange(len(exps_)))
        ax.set_yticklabels(exps_)

        cbar = fig.colorbar(im, ax=ax, shrink=0.9)
        cbar.set_label(f"{metric} − {baseline_exp}")

        self._apply_axes_limits(ax)

        if out_png is None and self.cfg.out_dir:
            out_png = self.cfg.out_dir / f"s2s_heatmapDiff_{metric}_{label}_vs_{baseline_exp}_{self.cfg.line_type}{'_' + self.cfg.tag if self.cfg.tag else ''}.png"
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")

        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig

    def plot_quadrant_heatmap(
        self,
        metric: str = "ACC",
        out_png: Path | None = None,
        show: bool = True,
        vmin: float | None = None,
        vmax: float | None = None,
        baseline_exp: str | None = None,   # if set, plot (metric - baseline)
        annotate: bool = False,
        cmap: str = "coolwarm",              # <- match your shared figure style
        center: float | None = None,       # if set, uses symmetric limits about center
        edgecolor: str = "white",
        linewidth: float = 0.6,
        guide: bool = True,                # <- add the triangle mapping guide box
        guide_loc: str = "upper left",     # 'upper left'|'upper right'|'lower left'|'lower right'
        guide_size: float = 0.16,          # fraction of axes width/height
        nlev: int = 9,
        xlabel: str = "Variable",
        ylabel: str = "Experiment",
        title_label: str = "", 
        fontz: int = 12, 
    ) -> plt.Figure:
        """
        One heatmap where each (exp, var) cell is split into 4 triangles for 4 lead bins.
    
        Layout:
          rows = experiments
          cols = variables
    
        Triangle mapping within each cell:
          UL = bin0 (lead_bins[0]), UR = bin1, LL = bin2, LR = bin3
        """
        bins = self.cfg.lead_bins
        labels = self.cfg.lead_bin_labels
        if len(bins) != 4 or len(labels) != 4:
            raise ValueError("plot_quadrant_heatmap expects exactly 4 lead bins/labels.")
    
        # variables (columns)
        vars_ = self.list_vars()
        if self.cfg.vars_include:
            vars_ = [v for v in vars_ if v in set(self.cfg.vars_include)]
        if not vars_:
            raise ValueError("No variables to plot after applying vars_include.")
    
        # experiments (rows)
        ds0 = self.open_skill(vars_[0])
        exps_ = [str(e) for e in ds0["exp"].values.tolist()]
    
        nvar, nexp = len(vars_), len(exps_)
    
        # V[var, exp, bin]
        V = np.full((nvar, nexp, 4), np.nan, dtype=float)
        for i, vname in enumerate(vars_):
            ds = self.open_skill(vname)
            lead = self._get_lead_days(ds)
    
            if metric not in ds:
                continue
    
            da = ds[metric]
            if "var" in da.dims:
                da = da.isel(var=0)
    
            for j, exp in enumerate(exps_):
                if exp not in da["exp"].values:
                    continue
                for k, (a, b) in enumerate(bins):
                    V[i, j, k] = float(self._bin_mean(da.sel(exp=exp), lead, a, b).values)
    
        title_suffix = ""
        if baseline_exp is not None:
            if baseline_exp not in exps_:
                raise ValueError(f"baseline_exp={baseline_exp} not found in exps={exps_}")
            j0 = exps_.index(baseline_exp)
            V = V - V[:, [j0], :]
            title_suffix = f" (diff vs {baseline_exp})"
    
        finite = V[np.isfinite(V)]
        if finite.size == 0:
            raise ValueError(f"All values are NaN for metric={metric}.")
    
        # --- color limits ---
        if center is not None:
            # symmetric around center
            if vmin is None or vmax is None:
                d = float(np.nanmax(np.abs(finite - center)))
                vmin, vmax = center - d, center + d
        else:
            if vmin is None:
                vmin = float(np.nanmin(finite))
            if vmax is None:
                vmax = float(np.nanmax(finite))
    
        # Discrete bins (e.g., 0.0..1.0 in steps of 0.1). Adjust as you like.
        base = plt.get_cmap(cmap)
        colors = base(np.linspace(0.08, 0.98, nlev))
        cm = ListedColormap(colors)
        bounds = np.linspace(vmin, vmax, nlev + 1)
        norm = BoundaryNorm(bounds, cm.N, clip=True)
        
        # figure sizing
        fig_w = max(9.0, 0.55 * nvar + 3.5)
        fig_h = max(4.0, 0.60 * nexp + 2.0)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    
        # --- draw triangles (row=exp=j, col=var=i) ---
        for j in range(nexp):        # rows (experiments)
            for i in range(nvar):    # cols (variables)
                x0, x1 = i, i + 1
                y0, y1 = j, j + 1
                xc, yc = (x0 + x1) / 2, (y0 + y1) / 2
    
                v_ul, v_ur, v_ll, v_lr = V[i, j, 0], V[i, j, 1], V[i, j, 2], V[i, j, 3]
                tris = [
                    ("UL", v_ul, [(x0, y1), (xc, yc), (x0, y0)]),
                    ("UR", v_ur, [(x1, y1), (xc, yc), (x0, y1)]),
                    ("LL", v_ll, [(x0, y0), (xc, yc), (x1, y0)]),
                    ("LR", v_lr, [(x1, y0), (xc, yc), (x1, y1)]),
                ]
    
                for _, val, pts in tris:
                    face = cm(norm(val)) if np.isfinite(val) else (1, 1, 1, 0)
                    poly = Polygon(
                        pts, closed=True,
                        facecolor=face,
                        edgecolor=edgecolor,
                        linewidth=linewidth,
                        joinstyle="miter",
                    )
                    ax.add_patch(poly)
    
                    if annotate and np.isfinite(val):
                        tx = sum(p[0] for p in pts) / 3
                        ty = sum(p[1] for p in pts) / 3
                        ax.text(tx, ty, f"{val:.2f}", ha="center", va="center", fontsize=6)
    
        # --- axes formatting ---
        ax.set_xlim(0, nvar)
        ax.set_ylim(0, nexp)
        ax.invert_yaxis()
    
        ax.set_xticks(np.arange(nvar) + 0.5)
        ax.set_xticklabels(vars_, rotation=30, ha="right", fontsize=fontz*0.95)
        ax.set_yticks(np.arange(nexp) + 0.5)
        ax.set_yticklabels(exps_, fontsize=fontz*0.95)
    
        ax.set_xlabel(xlabel, fontsize=fontz * 0.95, labelpad=6)
        ax.set_ylabel(ylabel, fontsize=fontz * 0.95, labelpad=6)
        
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=fontz * 0.85,
            length=4,
            width=1.0,
        )

        ax.set_title(
            f"{title_label}\n"
            f"UL={labels[0]}, UR={labels[1]}, LL={labels[2]}, LR={labels[3]}",
            fontsize = fontz*1.0
        )
    
        # --- colorbar ---
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cm)
        sm.set_array([])
        
        cbar = fig.colorbar(
            sm,
            ax=ax,
            shrink=0.9,
            pad=0.02,
            boundaries=bounds,
            ticks=bounds,
        )
        # --- label ---
        cbar.set_label(
            metric if baseline_exp is None else f"{metric} − {baseline_exp}",
            fontsize=fontz * 0.90,
        )
        # --- tick labels ---
        cbar.ax.tick_params(
            labelsize=fontz * 0.85,   # tick label font size
            length=4,
            width=1.0,
        )
        
        # (optional) force integer / clean formatting
        # cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

        # ----------------------------
        # Guide box (triangle mapping)
        # ----------------------------
        if guide:
            # ---- figure-fraction placement (outside main axes) ----
            # We place a square axes to the right of the main plot.
            # You can tweak these numbers to taste.
            # (x0, y0, w, h) are in figure coordinates [0..1]
            bbox = ax.get_position()  # Bbox in figure coords
            size = 0.16  # square size in figure fraction
            pad = 0.01   # gap from main axes
        
            # Put it near the top-right of the main axes, but OUTSIDE.
            x0 = min(bbox.x1 + pad, 1.0 - size)
            y0 = max(bbox.y1 - size, 0.02)
        
            iax = fig.add_axes([x0, y0, size, size])  # square
            iax.set_axis_off()
            iax.set_aspect("equal")
        
            # frame
            iax.add_patch(
                Polygon([(0, 0), (1, 0), (1, 1), (0, 1)],
                        closed=True, facecolor="white", edgecolor="black", linewidth=0.9)
            )
        
            # triangles + labels (square bin-map)
            xc, yc = 0.5, 0.5
            guide_tris = [
                ([(0, 1), (xc, yc), (0, 0)], "UL"),  # UL
                ([(1, 1), (xc, yc), (0, 1)], "UR"),  # UR
                ([(0, 0), (xc, yc), (1, 0)], "LL"),  # LL
                ([(1, 0), (xc, yc), (1, 1)], "LR"),  # LR
            ]
            for pts, lab in guide_tris:
                iax.add_patch(
                    Polygon(pts, closed=True, facecolor=(1, 1, 1, 0),
                            edgecolor="black", linewidth=0.6)
                )
                tx = sum(p[0] for p in pts) / 3
                ty = sum(p[1] for p in pts) / 3
                iax.text(tx, ty, lab, ha="center", va="center", fontsize=fontz*0.85)
        
            iax.text(0.5, 1.10, "Bin Map", ha="center", va="bottom", fontsize=fontz*0.85)

        #fig.tight_layout()
        fig.subplots_adjust(right=0.86)
        
        if out_png is None and self.cfg.out_dir:
            out_png = self.cfg.out_dir / self._out_name("quadheat", "ALLVARS", metric=metric)
        if out_png is not None:
            fig.savefig(out_png, dpi=self.cfg.dpi, bbox_inches="tight")
    
        if show:
            plt.show()
        else:
            plt.close(fig)
        return fig


    def make_all(
        self,
        do_curves: bool = True,
        do_bars: bool = True,
        do_shock: bool = True,
        curve_metrics: Sequence[str] = ("ACC",),
        bar_metric: str | None = None,
        shock_metric: str = "ACC",
        show: bool = False,
    ) -> None:
        vars_ = self.list_vars()
        for v in vars_:
            if do_curves:
                self.plot_skill_curves(v, metrics=curve_metrics, show=show)
            if do_bars:
                self.plot_week_binned_bars(v, metric=bar_metric, show=show)
            if do_shock:
                self.plot_shock_metric(v, metric=shock_metric, show=show)


In [3]:
if __name__ == "__main__":
    data_path = Path("/compyfs/zhan391/v3_dart_cda_scratch/diag_output/acc_rmse")
    fig_path = Path("./")
    fig_path.mkdir(parents=True, exist_ok=True)

    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            colors=["#7f7f7f", "#000000", "#1f77b4", "#d62728"],  # grey, black, blue, red
            period="201201-201203",
            season="Winter",
            init_month=1,
        ),
        "Jun2012": dict(
            models=["CAPT10-S1", "DART40-S1"],
            colors=["#000000", "#d62728"],  # black, red
            period="201206-201208",
            season="Summer",
            init_month=6,
        ),
    }

    group = "Jan2012"
    freq = "daily"
    run = "fc"
    obs = "ERA5"
    file_glob = f"s2s_skill_{group}_{freq}_{run}_{obs}_*.nc"

    exp_order = exp_list[group]["models"]
    exp_colors = dict(zip(exp_order, exp_list[group]["colors"]))  # <-- key fix

    var_dict = {
        "UBOT": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "VBOT": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "PSL": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "PRECT": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "U850": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "V850": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "T850": {"ACC": {"ylim": (0.9, 1.0), "xlim": (0, 60)}, "smooth_days": 0},
        "Z500": {"ACC": {"ylim": (0.9, 1.0), "xlim": (0, 60)}, "smooth_days": 0},
        "OMEGA500": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "U200": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
        "V200": {"ACC": {"ylim": (0.1, 0.6), "xlim": (0, 60)}, "smooth_days": 0},
    }

    vars_include = (
        "UBOT", "VBOT", "PSL", "PRECT", "U850", "V850", "T850", "Q850",
        "Z500", "OMEGA500", "U200", "V200",
    )

    lead_bins = ((1, 14), (15, 28), (29, 42), (43, 56))
    lead_bin_labels = ("Week1–2", "Week3–4", "Week5–6", "Week7–8")

    metric="ACC"
    vmin = 0.2
    vmax = 1.0
    guide = True
    guide_loc = "upper right"
    guide_size = 0.13
    edgecolor = "white"
    linewidth = 0.4
    show = True
    nlev = 8 
    xlabel = "Variable"
    ylabel = "" #"Experiment"
    fontz = 14
    title_label = "Anomaly Correlation Coefficient (ACC)"

    cfg = S2SSkillPlotConfig(
        skill_dir=data_path,
        file_glob=file_glob,
        out_dir=fig_path,
        exp_order=tuple(exp_order),
        exp_colors=exp_colors,                 # dict(exp->color)
        vars_include=tuple(vars_include),
        lead_bins=lead_bins,
        lead_bin_labels=lead_bin_labels,
        tag=f"{group}_{freq}_{run}_{obs}"
    )

    plotter = S2SSkillPlotter(cfg)

    # All-in-one triangle/quadrant heatmap
    plotter.plot_quadrant_heatmap(
        metric = metric,
        vmin = vmin, 
        vmax = vmax,
        guide = guide,
        guide_loc = guide_loc,
        guide_size = guide_size,
        edgecolor = edgecolor,
        linewidth = linewidth,
        xlabel = xlabel,
        ylabel = ylabel,
        fontz = fontz,
        nlev = nlev,
        title_label= title_label, 
        show=show,
    )
    

TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
if __name__ == "__main__":
    data_path = Path("/compyfs/zhan391/v3_dart_cda_scratch/diag_output/acc_rmse")
    fig_path = Path("./")
    fig_path.mkdir(parents=True, exist_ok=True)

    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            colors=["#7f7f7f", "#000000", "#1f77b4", "#d62728", ],
            period="201201-201203",
            season="Winter",
            init_month=1,
        ),
        "Jun2012": dict(
            models=["CAPT10-S1", "DART40-S1"],
            colors=["#000000", "#1f77b4", "#d62728", ],
            period="201206-201208",
            season="Summer",
            init_month=6,
        ),
    }
    
    group = "Jan2012"
    freq = "daily"
    run = "fc"
    obs = "ERA5"

    exp_order = exp_list[group]["models"]
    exp_colors = exp_list[group]["colors"]
    var_dict = {
        "UBOT": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "VBOT": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "PSL": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "PRECT": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "U850": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "V850": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "T850": {"ACC": [(0.9,1.0),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "Z500": {"ACC": [(0.9,1.0),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "OMEGA500": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "U200": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
        "V200": {"ACC": [(0.1,0.6),(0,60)], "RMSE": (0.0,None), "smooth_days": 0},
    }
    vars_include = (
        "UBOT", "VBOT", "PSL", "PRECT", "U850", "V850", "T850", "Q850",
        "Z500", "OMEGA500", "U200", "V200",
    )

    smooth_days = 0
    time_max = 60
    lead_bins = ((1, 14), (15, 28), (29, 42), (43, 56))
    lead_bin_labels = ("Week1–2", "Week3–4", "Week5–6", "Week7–8")

    var = "T850"
    shc_metric = "ACC"
    cur_metric = "ACC"
    bar_metric = "ACC"
    bar_use_quantiles = True
    line_type="ens_mean_std"          
    
    ylim = var_dict[var][bar_metric][0]
    xlim = var_dict[var][bar_metric][1]
    
    # Make sure we only pick up the selected group’s files
    file_glob = f"s2s_skill_{group}_{freq}_{run}_{obs}_*.nc"
    
    cfg = S2SSkillPlotConfig(
        skill_dir=data_path,
        file_glob=file_glob,
        out_dir=fig_path,
        exp_order=tuple(exp_order),
        exp_colors=exp_colors,
        vars_include=tuple(vars_include),
        smooth_days=smooth_days,
        time_max=time_max,
        lead_bins=lead_bins,
        lead_bin_labels=lead_bin_labels,
        bar_metric=bar_metric,
        bar_use_quantiles=bar_use_quantiles,
        # IMPORTANT: prevents overwriting outputs between Jan/Jun and records provenance in filenames
        tag=f"{group}_{freq}_{run}_{obs}",
        # Choose one:
        line_type=line_type,         # ensemble-mean deterministic skill
        xlim=xlim,
        ylim=ylim
    )

    plotter = S2SSkillPlotter(cfg)
        
    # Single variable quick looks
    plotter.plot_skill_curves(var, metrics=(cur_metric,), show=True)
    plotter.plot_week_binned_bars(var, metric=bar_metric, show=True)
    plotter.plot_shock_metric(var, metric=shc_metric, show=True)

    # Batch all
    # plotter.make_all(do_curves=True, do_bars=True, do_shock=True, show=False)
    